# ELECTRA + ScalarMix + DANN — Train / OOD split (your design)

**Train** (`education_level_judge`):
- Shrishti `train.csv`
- OneStop `ood_onestop.csv` (judge labels)
- RACE `ood_race-middle.csv` + `ood_race-high.csv` (judge labels)

**Val:** Shrishti `val.csv`

**OOD test** (from `judge_multi_corpus/clean_dataset/`):
- XSum, CoQA, WeeBit, CommonLit

Saves `best_model.pt` + `phase2_final.pt`; eval compares both on all OOD corpora.

In [ ]:
!pip install -q transformers scikit-learn torch pandas matplotlib seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_SHRISHTI_CLEAN = "/content/drive/MyDrive/BeyondFK/clean_dataset"
DRIVE_MULTI_CLEAN = "/content/drive/MyDrive/BeyondFK/trail/judge_multi_corpus/clean_dataset"
DRIVE_OUT_DIR = "/content/drive/MyDrive/BeyondFK/trail/electra_ose_race_train_multi_ood_dann"

MODEL_NAME = "google/electra-large-discriminator"
TEXT_COL = "full_text"
LABEL_COL = "education_level_judge"
SOURCE_COL = "source_dataset"

MAX_LEN = 512
BATCH_SIZE = 4
RNG_SEED = 42
LABEL_SMOOTHING = 0.1
DEDUPE_TEXT = True
EVAL_LABELS = [0, 1, 2]

GRL_LAMBDA_MAX = 0.15
DOMAIN_LOSS_ALPHA = 0.1
PHASE1_EPOCHS = 2
PHASE1_LR = 1e-3
PHASE2_EPOCHS = 1
PHASE2_LR = 2e-5
PARTIAL_FREEZE_LAYERS = 8
WARMUP_STEPS = 100

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

TRAIN_SOURCES = [
    (f"{DRIVE_SHRISHTI_CLEAN}/train.csv", "shrishti_train"),
    (f"{DRIVE_SHRISHTI_CLEAN}/ood_onestop.csv", "onestop_train"),
    (f"{DRIVE_SHRISHTI_CLEAN}/ood_race-middle.csv", "race_middle_train"),
    (f"{DRIVE_SHRISHTI_CLEAN}/ood_race-high.csv", "race_high_train"),
]
OOD_SOURCES = [
    (f"{DRIVE_MULTI_CLEAN}/xsum_2k.csv", "xsum_ood"),
    (f"{DRIVE_MULTI_CLEAN}/coqa_2k.csv", "coqa_ood"),
    (f"{DRIVE_MULTI_CLEAN}/weebit_500.csv", "weebit_ood"),
    (f"{DRIVE_MULTI_CLEAN}/commonlit_500.csv", "commonlit_ood"),
]

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import Tensor
from torch.nn import Parameter, ParameterList
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
CM_DIR = os.path.join(DRIVE_OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)
print("Device:", device)

In [ ]:
class ScalarMix(nn.Module):
    def __init__(self, mixture_size: int, trainable: bool = True) -> None:
        super().__init__()
        self.scalar_parameters = ParameterList(
            [Parameter(torch.zeros(1), requires_grad=trainable) for _ in range(mixture_size)]
        )
        self.gamma = Parameter(torch.ones(1), requires_grad=trainable)

    def forward(self, tensors: List[torch.Tensor]) -> torch.Tensor:
        w = torch.nn.functional.softmax(torch.cat([p for p in self.scalar_parameters]), dim=0)
        w = torch.split(w, 1)
        return self.gamma * sum(weight * t for weight, t in zip(w, tensors))


def grl_lambda_schedule(progress: float) -> float:
    progress = float(min(1.0, max(0.0, progress)))
    return GRL_LAMBDA_MAX * (2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx: Any, x: Tensor, lambda_: float) -> Tensor:
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> Tuple[Tensor, None]:
        return -ctx.lambda_ * grad_output, None


def apply_gradient_reversal(x: Tensor, lambda_: float) -> Tensor:
    return GradientReversalFunction.apply(x, float(lambda_))


def build_domain2id(sources: List[str]) -> Tuple[Dict[str, int], int]:
    unique = sorted(set(str(s) for s in sources))
    return {s: i for i, s in enumerate(unique)}, len(unique)


class DifficultyClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 3, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class DomainClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_domains: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_domains),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class ElectraScalarMixDANN(nn.Module):
    def __init__(self, model_name: str, num_classes: int, num_domains: int, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = int(self.encoder.config.hidden_size)
        n_layers = int(self.encoder.config.num_hidden_layers) + 1
        self.scalar_mix = ScalarMix(n_layers)
        self.dropout = nn.Dropout(dropout)
        self.difficulty_head = DifficultyClassifierHead(hidden, num_classes, dropout)
        self.domain_head = DomainClassifierHead(hidden, num_domains, dropout)

    def encode_pooled(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        mixed = self.dropout(self.scalar_mix(list(out.hidden_states)))
        mask = attention_mask.unsqueeze(-1).float()
        return (mixed * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def forward(self, input_ids: Tensor, attention_mask: Tensor, grl_lambda: float) -> Tuple[Tensor, Tensor]:
        pooled = self.encode_pooled(input_ids, attention_mask)
        diff_logits = self.difficulty_head(pooled)
        dom_logits = self.domain_head(apply_gradient_reversal(pooled, grl_lambda))
        return diff_logits, dom_logits

    def difficulty_logits_only(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        return self.difficulty_head(self.encode_pooled(input_ids, attention_mask))

    def freeze_encoder(self) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = True

    def freeze_encoder_except_top(self, n_layers: int) -> None:
        self.freeze_encoder()
        if hasattr(self.encoder, "encoder") and hasattr(self.encoder.encoder, "layer"):
            for layer in self.encoder.encoder.layer[-n_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True
        for p in self.scalar_mix.parameters():
            p.requires_grad = True


def combined_loss(diff_logits, dom_logits, labels, domains):
    ce_task = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ce_dom = nn.CrossEntropyLoss()
    diff_loss = ce_task(diff_logits, labels)
    dom_loss = ce_dom(dom_logits, domains)
    return diff_loss + DOMAIN_LOSS_ALPHA * dom_loss, diff_loss, dom_loss

print("DANN model OK")

In [ ]:
def _text_key(text: str) -> str:
    return hashlib.sha256(str(text).strip().encode("utf-8")).hexdigest()


def load_clean_csv(path: str, pool: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(p)
    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"{path}: need {TEXT_COL} and {LABEL_COL}")
    out = pd.DataFrame()
    out[TEXT_COL] = df[TEXT_COL].astype(str)
    out["label_str"] = df[LABEL_COL].astype(str).str.strip().str.lower()
    out["pool"] = pool
    if SOURCE_COL in df.columns:
        out[SOURCE_COL] = df[SOURCE_COL].astype(str)
    else:
        out[SOURCE_COL] = pool
    bad = ~out["label_str"].isin(label2id)
    if bad.any():
        print(f"  [{pool}] drop {bad.sum()} bad labels from {p.name}")
        out = out[~bad]
    out["label_id"] = out["label_str"].map(label2id).astype(int)
    return out.reset_index(drop=True)


def dedupe_train(df: pd.DataFrame) -> pd.DataFrame:
    if not DEDUPE_TEXT:
        return df
    priority = [p for _, p in TRAIN_SOURCES]
    rank = {p: i for i, p in enumerate(priority)}
    df = df.copy()
    df["_rk"] = df["pool"].map(lambda x: rank.get(x, 999))
    df["_key"] = df[TEXT_COL].map(_text_key)
    df = df.sort_values("_rk").drop_duplicates("_key", keep="first")
    return df.drop(columns=["_rk", "_key"]).reset_index(drop=True)


def assign_domain_ids(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    out = df.copy()
    out["domain_id"] = out[SOURCE_COL].map(domain2id)
    unk = out["domain_id"].isna()
    if unk.any():
        print(f"  [{split_name}] drop {unk.sum()} unknown domains")
        out = out[~unk].reset_index(drop=True)
    out["domain_id"] = out["domain_id"].astype(int)
    return out


train_parts = [load_clean_csv(path, pool) for path, pool in TRAIN_SOURCES]
df_train = dedupe_train(pd.concat(train_parts, ignore_index=True))
df_val = load_clean_csv(f"{DRIVE_SHRISHTI_CLEAN}/val.csv", "shrishti_val")

domain2id, num_domains = build_domain2id(
    pd.concat([df_train[SOURCE_COL], df_val[SOURCE_COL]]).tolist()
)
print(f"num_domains={num_domains}: {list(domain2id.keys())}")

df_train = assign_domain_ids(df_train, "train")
df_val = assign_domain_ids(df_val, "val")

ood_eval = {}
for path, name in OOD_SOURCES:
    ood_eval[name] = load_clean_csv(path, name)

print(f"\nTrain rows: {len(df_train)}")
print(df_train.groupby("pool")["label_str"].value_counts())
print(f"\nVal rows: {len(df_val)} | {dict(df_val['label_str'].value_counts())}")
for name, odf in ood_eval.items():
    print(f"OOD {name}: n={len(odf)} | judge={dict(odf['label_str'].value_counts())}")

manifest = {
    "train_pools": {p: int((df_train["pool"] == p).sum()) for p in df_train["pool"].unique()},
    "ood": {k: int(len(v)) for k, v in ood_eval.items()},
    "domain2id": domain2id,
}
with open(os.path.join(DRIVE_OUT_DIR, "data_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDANNDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray, domains: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)
        self.domains = domains.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "domain": torch.tensor(self.domains[idx], dtype=torch.long),
        }


def df_to_dann_loader(df: pd.DataFrame, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextDANNDataset(df[TEXT_COL].tolist(), df["label_id"].values, df["domain_id"].values),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
    )

train_loader = df_to_dann_loader(df_train, shuffle=True)
val_loader = df_to_dann_loader(df_val, shuffle=False)
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
model = ElectraScalarMixDANN(MODEL_NAME, num_classes=3, num_domains=num_domains).to(device)
best_path = os.path.join(DRIVE_OUT_DIR, "best_model.pt")
phase2_final_path = os.path.join(DRIVE_OUT_DIR, "phase2_final.pt")
history: List[Dict] = []
best_val_f1 = -1.0
global_step = 0
total_steps = (PHASE1_EPOCHS + PHASE2_EPOCHS) * len(train_loader)


@torch.no_grad()
def evaluate_val(grl_lambda: float = 0.0) -> Dict[str, float]:
    model.eval()
    ys, preds = [], []
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        diff_logits, _ = model(ids, mask, grl_lambda=grl_lambda)
        ys.extend(batch["label"].numpy().tolist())
        preds.extend(diff_logits.argmax(dim=1).cpu().numpy().tolist())
    return {
        "val_macro_f1": float(f1_score(ys, preds, labels=EVAL_LABELS, average="macro", zero_division=0)),
        "val_acc": float(accuracy_score(ys, preds)),
    }


def run_phase(phase_name: str, epochs: int, lr: float, grl_on: bool) -> None:
    global global_step, best_val_f1
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=min(WARMUP_STEPS, max(1, len(train_loader))),
        num_training_steps=max(1, epochs * len(train_loader)),
    )
    for epoch in range(epochs):
        model.train()
        run_diff, run_dom, n_batches = 0.0, 0.0, 0
        for batch in tqdm(train_loader, desc=f"{phase_name} ep{epoch+1}/{epochs}"):
            progress = global_step / max(total_steps - 1, 1)
            grl_l = grl_lambda_schedule(progress) if grl_on else 0.0
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            domains = batch["domain"].to(device)
            diff_logits, dom_logits = model(ids, mask, grl_lambda=grl_l)
            total, diff_loss, dom_loss = combined_loss(diff_logits, dom_logits, labels, domains)
            optimizer.zero_grad()
            total.backward()
            optimizer.step()
            scheduler.step()
            global_step += 1
            run_diff += diff_loss.item()
            run_dom += dom_loss.item()
            n_batches += 1
        metrics = evaluate_val()
        print(
            f"{phase_name} ep{epoch+1}: diff={run_diff/max(n_batches,1):.4f} "
            f"dom={run_dom/max(n_batches,1):.4f} val_f1={metrics['val_macro_f1']:.4f}"
        )
        if metrics["val_macro_f1"] > best_val_f1:
            best_val_f1 = metrics["val_macro_f1"]
            torch.save({"state_dict": model.state_dict(), "domain2id": domain2id}, best_path)
            print(f"  saved best val F1={best_val_f1:.4f}")
        history.append({"phase": phase_name, "epoch": epoch + 1, **metrics})


model.freeze_encoder()
run_phase("phase1_frozen_encoder", PHASE1_EPOCHS, PHASE1_LR, grl_on=False)
model.unfreeze_encoder()
model.freeze_encoder_except_top(PARTIAL_FREEZE_LAYERS)
run_phase("phase2_dann", PHASE2_EPOCHS, PHASE2_LR, grl_on=True)
torch.save({"state_dict": model.state_dict(), "domain2id": domain2id, "phase": "phase2_dann_final"}, phase2_final_path)
print(f"Saved {phase2_final_path} | best val F1={best_val_f1:.4f}")

In [ ]:
def save_confusion_matrix(y_true, y_pred, title, out_path, labels):
    names = [id2label[i] for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def load_checkpoint(path: str) -> None:
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])


@torch.no_grad()
def predict_texts(texts: List[str], batch_size: int = 16) -> np.ndarray:
    model.eval()
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="predict", leave=False):
        enc = tokenizer(
            texts[i : i + batch_size], truncation=True, max_length=MAX_LEN,
            padding=True, return_tensors="pt",
        )
        logits = model.difficulty_logits_only(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


def eval_split(name, y_true, y_pred, tag, verbose=True):
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    f1_all = f1_score(y_true, y_pred, labels=EVAL_LABELS, average="macro", zero_division=0)
    f1_pres = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    if verbose:
        print(f"\n{'='*60}\n[{tag}] {name}\n{'='*60}")
        print(classification_report(y_true, y_pred, labels=EVAL_LABELS, target_names=[id2label[i] for i in EVAL_LABELS], zero_division=0))
        print(f"Macro-F1: {f1_all:.4f} | present-only: {f1_pres:.4f}")
    save_confusion_matrix(y_true, y_pred, f"{tag} — {name}", os.path.join(CM_DIR, f"cm_{tag}_{name}.png"), EVAL_LABELS)
    return {"checkpoint": tag, "corpus": name, "n": len(y_true), "macro_f1_3class": float(f1_all), "macro_f1_present": float(f1_pres)}


CHECKPOINTS = [("phase1_best_val", best_path)]
if os.path.exists(phase2_final_path):
    CHECKPOINTS.append(("phase2_dann_final", phase2_final_path))

all_rows = []
for tag, path in CHECKPOINTS:
    print(f"\n{'#'*60}\nOOD eval: {tag}\n{'#'*60}")
    load_checkpoint(path)
    for name, odf in ood_eval.items():
        y_true = odf["label_id"].values
        y_pred = predict_texts(odf[TEXT_COL].tolist())
        all_rows.append(eval_split(name, y_true, y_pred, tag))

comparison = pd.DataFrame(all_rows)
pivot = comparison.pivot(index="corpus", columns="checkpoint", values="macro_f1_3class")
print("\nOOD macro-F1:\n", pivot.to_string())
print("\nMean OOD macro-F1:", comparison.groupby("checkpoint")["macro_f1_3class"].mean().to_string())

comparison.to_csv(os.path.join(DRIVE_OUT_DIR, "eval_comparison.csv"), index=False)
pivot.reset_index().to_csv(os.path.join(DRIVE_OUT_DIR, "eval_pivot_macro_f1.csv"), index=False)
with open(os.path.join(DRIVE_OUT_DIR, "eval_results.json"), "w") as f:
    json.dump({"history": history, "evaluations": all_rows, "best_val_f1": best_val_f1}, f, indent=2)
print("\nSaved to", DRIVE_OUT_DIR)